In [1]:
from cmath import nan

import numpy as np
import yfinance as yf
import pandas as pd
import datetime as dt

import hist_volatility
from riskfree_and_spot import *
from hist_volatility import *
from blackscholes import *
from binomial import *
from imp_volatility import *

In [2]:
option_style = "american" #european / american
valuation_date = "2026-08-04" #YYYY-MM-DD
ticker = "MSFT"
expiration = "2026-08-05"
strike = 487.5
option_type = "Call" # call / put
volatility_method = "historical" #historical / (implied /not yet implemented/)
hist_vol_lookback = 1 #in years

#BINOMIAL
n_for_binomial = 1500 #t0 + n steps

In [3]:
spot_price = get_spot_price_data(ticker=ticker, valuation_date=valuation_date)
t_t_m = option_tenor_calc(val_date=valuation_date, exp_date=expiration)
risk_free_rate = get_risk_free_rate(val_date=valuation_date, exp_date=expiration)
vol = get_volatility(ticker=ticker, val_date=valuation_date, lookback=hist_vol_lookback)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [4]:
# #UPDATED_OPTION_CHAIN_QUARY
#
# ticker_obj = yf.Ticker(ticker)
# exps = ticker_obj.options
# all_data = []
#
# for date in exps:
#     option_chain = ticker_obj.option_chain(date)
#     if option_type.lower() =="call":
#         type_chain = option_chain.calls
#     elif option_type.lower() =="put":
#         type_chain = option_chain.puts
#     else:
#         raise ValueError("option_style must be either 'call' or 'put'")
#     type_chain["expiration"] = date
#
#     all_data.append(type_chain)
#
# all_data = pd.concat(all_data, ignore_index = True)
# all_data

In [30]:
raw_data = pd.read_csv("../data/option_data.csv")

In [31]:
#TENOR_CALC

tenor_list = []

for i, row in raw_data.iterrows():
    exp_date = row["expiration"]
    tenor = option_tenor_calc(val_date=valuation_date, exp_date=exp_date)
    tenor_list.append(tenor)

raw_data["tenor"] = tenor_list

In [32]:
#RF_RATE_CALC

exp_list = raw_data["expiration"].unique()

rf_for_exp_dict = {
    exp: get_risk_free_rate(val_date=valuation_date, exp_date=exp)
    for exp in exp_list
}

raw_data["rf_rate"] = raw_data["expiration"].map(rf_for_exp_dict)


In [33]:
#MID_PRICE_CALC

mid_price_list = []

for i, row in raw_data.iterrows():
    mid = (row["bid"] + row["ask"]) / 2
    mid_price_list.append(mid)

raw_data["mid_price"] = mid_price_list

In [34]:
#SPREAD_CALC

spread_list = []
for i, row in raw_data.iterrows():
    spread = row["ask"] - row["bid"]
    spread_list.append(spread)

raw_data["spread"] = spread_list

In [35]:
#LIQUIDITY_CUT
raw_data_lq_filter = raw_data[
    (raw_data["bid"] > 0) &
    (raw_data["ask"] > 0) &
    (raw_data["volume"] > 3) &
    (raw_data["openInterest"] > 0) &
    (raw_data["strike"] > 0.7 * spot_price) &
    (raw_data["strike"] < 1.3 * spot_price) &
    ((raw_data["spread"] < (0.3 * raw_data["mid_price"])) |
    (raw_data["spread"] <= 0.05))
]

raw_data_lq_filter


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration,tenor,rf_rate,mid_price,spread
15,MSFT260807C00345000,2026-08-06 17:12:50+00:00,345.0,150.82,153.40,157.15,34.580010,29.748804,4.0,89,2.339848,True,REGULAR,USD,2026-08-07,0.008219,0.037800,155.275,3.75
16,MSFT260807C00350000,2026-08-05 19:48:29+00:00,350.0,141.00,148.40,152.25,0.000000,0.000000,5.0,208,2.297856,True,REGULAR,USD,2026-08-07,0.008219,0.037800,150.325,3.85
21,MSFT260807C00365000,2026-08-06 19:49:01+00:00,365.0,133.10,133.40,137.30,6.860008,5.434101,9.0,610,2.075200,True,REGULAR,USD,2026-08-07,0.008219,0.037800,135.350,3.90
22,MSFT260807C00367500,2026-08-06 19:33:23+00:00,367.5,131.97,130.95,134.80,8.599998,6.970900,6.0,26,2.051763,True,REGULAR,USD,2026-08-07,0.008219,0.037800,132.875,3.85
23,MSFT260807C00370000,2026-08-06 19:53:42+00:00,370.0,127.73,128.40,131.90,6.850006,5.666782,162.0,424,1.849610,True,REGULAR,USD,2026-08-07,0.008219,0.037800,130.150,3.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1603,MSFT281215C00615000,2026-08-05 13:39:08+00:00,615.0,75.65,79.50,82.85,0.000000,0.000000,10.0,94,0.394850,False,REGULAR,USD,2028-12-15,2.367123,0.042159,81.175,3.35
1605,MSFT281215C00625000,2026-08-05 16:41:48+00:00,625.0,73.25,76.75,80.30,0.000000,0.000000,37.0,975,0.394858,False,REGULAR,USD,2028-12-15,2.367123,0.042159,78.525,3.55
1606,MSFT281215C00630000,2026-08-06 19:59:16+00:00,630.0,77.29,76.25,78.35,6.290001,8.859157,13.0,327,0.392546,False,REGULAR,USD,2028-12-15,2.367123,0.042159,77.300,2.10
1607,MSFT281215C00635000,2026-07-31 17:21:22+00:00,635.0,54.11,73.50,77.05,0.000000,0.000000,4.0,69,0.392294,False,REGULAR,USD,2028-12-15,2.367123,0.042159,75.275,3.55


In [36]:
#NO_ARGBITRAGE_CHECK

no_arbitrage_flag_list = []

for i, row in raw_data_lq_filter.iterrows():
    k = row["strike"]
    rf = row["rf_rate"]
    t = row["tenor"]
    mid = row["mid_price"]
    lower_bound = max(0, spot_price - (k * np.exp(-rf * t)))
    upper_bound = spot_price
    if mid > lower_bound and mid < upper_bound:
        arbitrage_flag = True
    else:
        arbitrage_flag = False
    no_arbitrage_flag_list.append(arbitrage_flag)

raw_data_lq_filter["no_arbitrage_flag"] = no_arbitrage_flag_list

raw_data_lq_filter

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expiration,tenor,rf_rate,mid_price,spread,no_arbitrage_flag
15,MSFT260807C00345000,2026-08-06 17:12:50+00:00,345.0,150.82,153.40,157.15,34.580010,29.748804,4.0,89,2.339848,True,REGULAR,USD,2026-08-07,0.008219,0.037800,155.275,3.75,True
16,MSFT260807C00350000,2026-08-05 19:48:29+00:00,350.0,141.00,148.40,152.25,0.000000,0.000000,5.0,208,2.297856,True,REGULAR,USD,2026-08-07,0.008219,0.037800,150.325,3.85,True
21,MSFT260807C00365000,2026-08-06 19:49:01+00:00,365.0,133.10,133.40,137.30,6.860008,5.434101,9.0,610,2.075200,True,REGULAR,USD,2026-08-07,0.008219,0.037800,135.350,3.90,True
22,MSFT260807C00367500,2026-08-06 19:33:23+00:00,367.5,131.97,130.95,134.80,8.599998,6.970900,6.0,26,2.051763,True,REGULAR,USD,2026-08-07,0.008219,0.037800,132.875,3.85,True
23,MSFT260807C00370000,2026-08-06 19:53:42+00:00,370.0,127.73,128.40,131.90,6.850006,5.666782,162.0,424,1.849610,True,REGULAR,USD,2026-08-07,0.008219,0.037800,130.150,3.50,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1603,MSFT281215C00615000,2026-08-05 13:39:08+00:00,615.0,75.65,79.50,82.85,0.000000,0.000000,10.0,94,0.394850,False,REGULAR,USD,2028-12-15,2.367123,0.042159,81.175,3.35,True
1605,MSFT281215C00625000,2026-08-05 16:41:48+00:00,625.0,73.25,76.75,80.30,0.000000,0.000000,37.0,975,0.394858,False,REGULAR,USD,2028-12-15,2.367123,0.042159,78.525,3.55,True
1606,MSFT281215C00630000,2026-08-06 19:59:16+00:00,630.0,77.29,76.25,78.35,6.290001,8.859157,13.0,327,0.392546,False,REGULAR,USD,2028-12-15,2.367123,0.042159,77.300,2.10,True
1607,MSFT281215C00635000,2026-07-31 17:21:22+00:00,635.0,54.11,73.50,77.05,0.000000,0.000000,4.0,69,0.392294,False,REGULAR,USD,2028-12-15,2.367123,0.042159,75.275,3.55,True
